In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score, KFold
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
mlb_pitchers = pd.read_csv('stored_datasets/All Pitchers Cleaned.csv')
mlb_pitchers

,Player,Debut Year,Debut Age,Retirement Year,Retirement Age,Career Length,Wins,Losses,Win Percentage,Total Decisions,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Hall of Fame
0,Cy Young,1890,23,1911,44,21,511,315,0.619,826,...,29565,138,2.84,1.130,8.7,0.2,1.5,3.4,2.30,1
1,Pud Galvin,1875,18,1892,35,17,365,310,0.541,675,...,25415,107,2.96,1.191,9.6,0.2,1.1,2.7,2.43,1
2,Walter Johnson,1907,19,1927,39,20,417,279,0.599,696,...,23415,147,2.38,1.061,7.5,0.1,2.1,5.3,2.57,1
3,Phil Niekro,1964,25,1987,48,23,318,274,0.537,592,...,22677,115,3.62,1.268,8.4,0.8,3.0,5.6,1.85,1
4,Nolan Ryan,1966,19,1993,46,27,324,292,0.526,616,...,22575,112,2.97,1.247,6.6,0.5,4.7,9.5,2.04,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,1919,23,1928,32,9,49,80,0.380,129,...,4453,90,3.89,1.518,10.3,0.3,3.4,2.8,0.82,0
1211,Sammy Ellis,1962,21,1969,28,7,63,58,0.521,121,...,4296,88,3.90,1.340,8.7,1.1,3.4,6.1,1.79,0
1212,Lil Stoner,1922,23,1931,32,9,50,57,0.467,107,...,4466,87,4.13,1.548,10.6,0.6,3.4,2.7,0.80,0
1213,Johnny Humphries,1938,23,1946,31,8,52,63,0.452,115,...,4342,97,3.80,1.394,9.2,0.4,3.4,2.8,0.85,0


In [5]:
svc = SVC()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(max_iter=1000, random_state=42))
])

# Old version with no K-Fold or Gridsearch

In [5]:
svc = SVC()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svc = SVC()
svc.fit(X_train, y_train)

SVC()

In [13]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.svm import SVC
import numpy as np
import pandas as pd

# Define model with probability enabled
svc = SVC(probability=True, random_state=42)

# Cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Generate predicted classes (out-of-fold)
y_pred_all = cross_val_predict(svc, X, y, cv=kf)

# Generate predicted probabilities (out-of-fold)
y_proba_all = cross_val_predict(svc, X, y, cv=kf, method='predict_proba')

# If binary classification, pick the probability of the predicted class
y_pred_indices = np.array([list(np.unique(y)).index(p) for p in y_pred_all])
confidence_scores = y_proba_all[np.arange(len(y_proba_all)), y_pred_indices]

# Build results dataframe
svc_cv_results = pd.DataFrame()
svc_cv_results['Player'] = mlb_pitchers['Player']
svc_cv_results['Hall of Fame'] = y
svc_cv_results['Prediction'] = y_pred_all
svc_cv_results['Prediction Correct'] = (svc_cv_results['Hall of Fame'] == svc_cv_results['Prediction']).astype(int)
svc_cv_results['Prediction Confidence'] = confidence_scores

svc_cv_results

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.812674
1,Pud Galvin,1,1,1,0.993500
2,Walter Johnson,1,1,1,0.991771
3,Phil Niekro,1,1,1,0.988360
4,Nolan Ryan,1,1,1,0.997005
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.955076
1211,Sammy Ellis,0,0,1,0.959009
1212,Lil Stoner,0,0,1,0.960723
1213,Johnny Humphries,0,0,1,0.954236


In [11]:
svc_cv_results['Prediction Correct'].value_counts()

Prediction Correct
1    1152
0      63
Name: count, dtype: int64

In [14]:
svc_cv_results.to_csv('Results SVC no GS.csv', index=False)